Ejemplo 2. Procesamiento de lenguaje natural (NLP)

Para este ejemplo usaremos el conjunto de datos imdb para clasificar reseñas como positivas o negativas.

El primer paso es importar las librerías que usaremos.


In [ ]:
import numpy as np
import keras
from keras import layers

Ahora cargamos los datos y la única transformación que se llevará a cabo es la conversión de los textos en secuencias. Esto es necesario porque usaremos el modelo de capas LSTM.

In [ ]:
max_features = 20000
maxlen = 200

(x_train, y_train), (x_val_test, y_val_test) = keras.datasets.imdb.load_data(
    num_words=max_features
)
x_val = x_val_test[:int (len(x_val_test)/2)]
y_val = y_val_test[:int (len(y_val_test)/2)]
x_test = x_val_test[int (len(x_val_test)/2):]
y_test = y_val_test[int (len(y_val_test)/2):]
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")
print(len(x_val), "Test sequences")

x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.utils.pad_sequences(x_val, maxlen=maxlen)
x_test = keras.utils.pad_sequences(x_test, maxlen=maxlen)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
25000 Training sequences
12500 Validation sequences
12500 Test sequences


A continuación, una vez que temenos los datos en el formato correcto, creamos el modelo de la siguiente manera:

In [ ]:
inputs = keras.Input(shape=(None,), dtype="int32")
x = layers.Embedding(max_features, 128)(inputs)
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

Y finalmente entrenamos y evaluamos el modelo:

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=32, epochs=2, validation_data=(x_val, y_val))

score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 462s 580ms/step - accuracy: 0.7461 - loss: 0.4862 - val_accuracy: 0.8517 - val_loss: 0.3700
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 539s 627ms/step - accuracy: 0.9209 - loss: 0.2094 - val_accuracy: 0.8635 - val_loss: 0.3330
Test loss: 0.3107074797153473
Test accuracy: 0.8727999925613403
